In [ ]:
import json
import sys
from datetime import datetime, timezone
from pathlib import Path

REPO_ROOT = Path.cwd().parent
DB_PATH = REPO_ROOT / "database" / "CSCAF_public.db"
COURSE_FOLDER = REPO_ROOT / "raw_data" / "training_courses"
REPORT_FOLDER = REPO_ROOT / "data_load" / "load_reports"
PRIMARY_VARIANT = "overview_learn_takeaways"

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

assert DB_PATH.exists(), "run 01_load_ecsf_to_database.ipynb and 06_load_industry_trainings.ipynb first"

In [ ]:
from cscaf.loading import TrainingLoader
from cscaf.storage import Storage
from cscaf.study import Study
from cscaf.text import TextPipeline

study = Study.from_config()
config = study.config
storage = Storage.of(study, read_only=False)
run = study.run_record()
RUN = storage.runs.ensure_run(run)
loader = TrainingLoader(study, storage, TextPipeline(study))
print(f"run             : {RUN}")
print(f"embedding model : {config['model']['name']} @ {config['model'].get('revision')}")
print(f"segmentation    : {config['segmentation']['model']} @ {config['segmentation'].get('model_revision')}")
print(f"preprocessing   : {'on' if config['segmentation'].get('preprocessing', {}).get('enabled') else 'OFF'}")
print(f"database        : {storage.path.relative_to(REPO_ROOT)}")

In [ ]:
segmented = loader.segment(RUN)
print(f"segmented {segmented['courses']} course texts -> {segmented['sentences']} sentences "
      f"({segmented['repeated']} repeated, {segmented['dropped']} units dropped by the cleaning rules)")
print(f"\n{'code':8s} {'variant':26s} {'sent':>5s}  roles the provider declares")
for training in storage.trainings.trainings(RUN):
    print(f"{training['code']:8s} {training['variant']:26s} {training['sentences']:5d}  "
          f"{', '.join(training['roles'])}")

In [ ]:
scored = loader.similarities(RUN)
print(f"scored {scored['courses']} course texts -> {scored['pairs']:,} pairs, "
      f"{scored['top_1_rows']:,} top-1 rows")
texts = len(storage.trainings.trainings(RUN))
primary = storage.similarities.count_training_best_matches(RUN)
print(f"primary rows {primary} = {texts} texts x 150 statements")
assert primary == texts * 150, primary

In [ ]:
REPORT_FOLDER.mkdir(parents=True, exist_ok=True)
output = REPORT_FOLDER / f"industry_training_{RUN}.json"

trainings = storage.trainings.trainings(RUN)
variants = sorted({t["variant"] for t in trainings})
record = {
    "created_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "run": RUN,
    "run_info": run,
    "embedding_model": config["model"]["name"],
    "embedding_revision": config["model"].get("revision"),
    "segmentation_model": config["segmentation"]["model"],
    "segmentation_revision": config["segmentation"].get("model_revision"),
    "preprocessing": config["segmentation"].get("preprocessing"),
    "primary_variant": PRIMARY_VARIANT,
    "variants": variants,
    "source_folder": str(COURSE_FOLDER),
    "segmentation": segmented,
    "similarity": scored,
    "trainings": trainings,
    "sentences_by_variant": {variant: sorted(t["sentences"] for t in trainings if t["variant"] == variant)
                             for variant in variants},
}
output.write_text(json.dumps(record, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"written to {output.relative_to(REPO_ROOT)}")
for variant, counts in record["sentences_by_variant"].items():
    print(f"  {variant:26s} {counts}")